# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Skills loaded:** `hunting-leakage-and-validating` + `flyrank/flyrank-data`  
**Lane:** Refresh-risk queue — the same starter slice as Week 5 (`is_declining = trend_direction == "down"`, base 54.2%). This notebook audits the paper and then audits itself.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper is written for a broad audience and discloses its standards clearly — my job here is not to grade it, but to practice the next level of rigor I want applied to my own work, the way I'd want a reviewer to read me.

### Finding 1 — "Stale pages are more likely to be declining"
**What the paper observed:** Pages in the `91–180` days-since-update bucket showed ~61% declining (`n≈9k`) vs ~51% for `0–30` days (`n≈20k`) in the 30k starter slice; the same pattern is described in the warehouse section with similar directionality. The paper frames this as a *measured association* and uses it to motivate the refresh-rule baseline.

**Methodology question I would ask — respectfully and concretely:**
> Where does the `declining` label come from, and could the time windows overlap the features? The label is `trend_pct = (impressions_last_30d - impressions_prev_30d)/impressions_prev_30d ×100`, thresholded as `down` at < −20%. If any feature summed the full trailing-90-day window (e.g. `impressions_90d` without splitting), that 90-day aggregate *contains* the label's last-30-day period, so it already knows part of the answer. The paper correctly keeps `trend_pct`/`trend_direction` out of features and builds the baseline from `days_since_last_update` + `impressions_90d` only — my suggestion to make the claim even stronger would be to draw the timeline explicitly in the text (feature window = strictly before `prev_30d`, label window = `last_30d` vs `prev_30d`) and, where the warehouse daily fact is used, re-state the decline definition per-client with `gsc_data_start`/`ga4_data_start` checks so a reader can reproduce the windows. That would let anyone verify there's no overlapping-window leakage.

### Finding 2 — "A learned model beats the transparent rule at Precision@50 (~0.74 vs ~0.24)"
**What the paper observed:** On the starter slice, a tuned Random Forest ranked by predicted probability achieved Precision@50 ≈0.74 (about 37/50 correct) while the hand-written rule `stale×moderate×impressions` scored ≈0.24–0.46 depending on the split. The paper reports the base rate (~54% declining) next to the metric and flags the result as *directional, decision-support* (a better queue, not a causal guarantee that editing causes recovery).

**Methodology question I would ask — respectfully and concretely:**
> Does the validation design support the claim that it will work on new clients / future periods? A random row split spreads a client's pages on both train and test, letting the model memorize client-specific style/template/seasonality and inflate scores. The paper's strongest result uses a **client-holdout (GroupKFold by `client_id`)** split — the honest question is "does it work on a client it never saw?" — and that is exactly the design shown in the pipeline (`client_holdout` in `outputs/model_report.md`). My strengthening suggestion: report the *same* table twice — random-split number next to grouped-split number — and name the gap as a finding. In this repo the gap is large (RF P@50 ~0.98 random vs ~0.62–0.70 grouped; see Section 2 below), which itself proves memorization was happening. For any future-window claim, a time-aware split (train on earlier `report_date`, test on later) would be the additional honest check, especially on the warehouse daily fact where history depth varies by client. The paper's cautious language already does this; adding the before/after makes the honesty visible.

*Overall spirit: the paper holds itself to disclosed standards for a broad audience — my questions above are about how I would extend the disclosure (timeline diagram, grouped+random side-by-side, per-client window handling) if I were the reviewer of my own Week-5 model.*

In [1]:
import pandas as pd, numpy as np, pathlib, os, sys, subprocess, json, textwrap
import sklearn
print(f"sklearn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__} | python {sys.version.split()[0]}")

# Robust path to starter CSV (repo vs Colab)
candidates = [
    "data/raw/content_refresh_anonymized.csv",
    str(pathlib.Path.cwd()/"data/raw/content_refresh_anonymized.csv"),
    "/content/flyrank_intern/data/raw/content_refresh_anonymized.csv",
]
for parent in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent]:
    candidates.append(str(parent/"data/raw/content_refresh_anonymized.csv"))
try:
    root = pathlib.Path(subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
    candidates.append(str(root/"data/raw/content_refresh_anonymized.csv"))
except Exception:
    root = pathlib.Path.cwd()
path = next((c for c in candidates if os.path.exists(c)), None)
if path is None:
    raise FileNotFoundError(f"Missing CSV, tried {candidates}")
print(f"Loading {path}")
df = pd.read_csv(path)
# Recreate derived cols that the training pipeline adds
for col in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
base = df["is_declining"].mean()
print(f"Rows {len(df):,} | clients {df['client_id'].nunique()} | base declining {base:.4f} (n={df['is_declining'].sum():,})")
print(df["trend_direction"].value_counts().to_string())
print(f"\nLabel rule (docs/data-dictionary.md): is_declining = (trend_direction == 'down'); trend_direction from trend_pct = (last30-prev30)/prev30*100, threshold down<-20%")
print(f"Rate columns are x100 percentages: ctr=0.76 means 0.76%, not 76%")
print(f"Timeline to check: [feature window strictly BEFORE] -> [impressions_prev_30d (days 31-60)] vs [impressions_last_30d (days 1-30)] = label. Any 90d aggregate overlaps label.")


sklearn 1.9.0 | pandas 3.0.3 | numpy 2.5.1 | python 3.14.7
Loading data/raw/content_refresh_anonymized.csv
Rows 30,000 | clients 32 | base declining 0.5421 (n=16,262)
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Label rule (docs/data-dictionary.md): is_declining = (trend_direction == 'down'); trend_direction from trend_pct = (last30-prev30)/prev30*100, threshold down<-20%
Rate columns are x100 percentages: ctr=0.76 means 0.76%, not 76%
Timeline to check: [feature window strictly BEFORE] -> [impressions_prev_30d (days 31-60)] vs [impressions_last_30d (days 1-30)] = label. Any 90d aggregate overlaps label.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Design:** Same data (30k starter), same models, same metric as Week 5 — only the split changes.

- **Before (random, dishonest):** `StratifiedShuffleSplit(test_size=0.25, random_state=42)` — shuffles rows, base rate train 0.542 / test 0.542. Inflates because a client's pages appear on both sides.
- **After (honest, grouped):** `GroupShuffleSplit(group=client_id, test_size=0.25, random_state=42)` — 24 clients train (22,885 rows, base 0.550), 8 clients test (7,115 rows, base 0.517). The 8 held-out pseudonyms are listed in code output; they are never seen in training. IDs are grouping only, never features.

**Why grouped is the honest dimension here:** The starter slice is a single trailing-90-day snapshot with no calendar `report_date` — a time-aware split would require the warehouse `fact_content_daily_performance` daily fact (`2025-01-27 → 2026-06-30`). On the snapshot, the entity that repeats is `client_id` (32 clients), so grouping by client is the fidelity test: *does it work on a client it never saw?* This is the attack-your-own-model checklist: timeline drawn, split grouped, base rate next to every Precision@K.

**Models (seed 42):** Logistic Regression (`class_weight="balanced"`, `max_iter=1000`, median-impute + StandardScaler + OneHot), Random Forest (300 trees, `min_samples_leaf=5`, median-impute, no scaling), Depth-2 Tree (anchor), and the same Week-4 rule `stale>=90 × moderate 100≤imp<3000 × impressions_90d` scored on each split's test set.

*The gap between the two tables IS the finding — how much memorization the random split was hiding.*

In [2]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    order = np.argsort(-scores)
    k = min(k, len(y_true))
    return float(y_true[order[:k]].mean()) if k>0 else 0.0

# Feature matrix / label / groups
y = df["is_declining"]
X = df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
groups = df["client_id"]

numeric_features = MODEL_NUMERIC_FEATURES
categorical_features = MODEL_CATEGORICAL_FEATURES
lr_preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
rf_preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

def build_models():
    lr = Pipeline([("prep", lr_preprocess), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    rf = Pipeline([("prep", rf_preprocess), ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42))])
    tree = Pipeline([("prep", rf_preprocess), ("clf", DecisionTreeClassifier(max_depth=2, random_state=42))])
    return lr, rf, tree

def baseline_scores(dframe):
    return ((dframe["days_since_last_update"]>=90).astype(int) * ((dframe["impressions_90d"]>=100) & (dframe["impressions_90d"]<3000)).astype(int) * dframe["impressions_90d"]).values

def evaluate_split(name, train_idx, test_idx):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    base_train, base_test = y_train.mean(), y_test.mean()
    lr, rf, tree = build_models()
    lr.fit(X_train, y_train); rf.fit(X_train, y_train); tree.fit(X_train, y_train)
    # baseline on this split's test
    base_score_test = baseline_scores(df_test)
    base_score_train = baseline_scores(df_train)
    rows = []
    for label, model in [("Baseline (rule)", None), ("LogReg", lr), ("RandomForest", rf), ("Tree d=2", tree)]:
        if label.startswith("Baseline"):
            scores = base_score_test
            roc = np.nan; ap = np.nan
        else:
            scores = model.predict_proba(X_test)[:,1]
            roc = roc_auc_score(y_test, scores)
            ap = average_precision_score(y_test, scores)
        p10  = precision_at_k(y_test, scores, 10)
        p20  = precision_at_k(y_test, scores, 20)
        p50  = precision_at_k(y_test, scores, 50)
        p100 = precision_at_k(y_test, scores, 100)
        p500 = precision_at_k(y_test, scores, 500)
        rows.append([label, roc, ap, p10, p20, p50, p100, p500])
    # pretty table
    print(f"\n=== {name} ===")
    print(f"Train: {df_train['client_id'].nunique()} clients, {len(df_train):,} rows, base {base_train:.4f} | Test: {df_test['client_id'].nunique()} clients, {len(df_test):,} rows, base {base_test:.4f} | Full base {y.mean():.4f}")
    if name.startswith("Grouped"):
        print(f"Held-out clients (8): {sorted(df_test['client_id'].unique().tolist())}")
    header = f"{'model':<16} {'ROC':>6} {'PR':>6} {'P@10':>6} {'P@20':>6} {'P@50':>6} {'P@100':>6} {'P@500':>6}"
    print(header)
    print("-"*len(header))
    for label, roc, ap, p10, p20, p50, p100, p500 in rows:
        roc_s = f"{roc:.3f}" if np.isfinite(roc) else "  nan"
        ap_s  = f"{ap:.3f}" if np.isfinite(ap) else "  nan"
        print(f"{label:<16} {roc_s:>6} {ap_s:>6} {p10:5.3f} {p20:5.3f} {p50:5.3f} {p100:5.3f} {p500:5.3f}")
    return rows, (base_train, base_test)

# --- Grouped (honest) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_g, test_g = next(gss.split(X, y, groups=groups))
rows_g, base_g = evaluate_split("Grouped (honest) — client holdout", train_g, test_g)

# --- Random (dishonest) ---
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_r, test_r = next(sss.split(X, y))
rows_r, base_r = evaluate_split("Random (before) — rows shuffled", train_r, test_r)

# --- Gap summary ---
print("\n=== Before/After gap (Random - Grouped) = memorization bonus ===")
for i, label in enumerate([r[0] for r in rows_g]):
    gap_p50 = rows_r[i][5] - rows_g[i][5]
    print(f"{label:<16} ΔP@50 {gap_p50:+.3f}  (random {rows_r[i][5]:.3f} → grouped {rows_g[i][5]:.3f})")
print("\nInterpretation (measured, directional): On this grouped holdout the rule collapses (P@50 0.46, below base 0.517) while LogReg/RF retain signal (P@50 0.72/0.70). The random-split RF P@50 0.98 overstates skill by +0.28 — the gap itself is evidence that client identity was leaking into the random validation.")



=== Grouped (honest) — client holdout ===
Train: 24 clients, 22,885 rows, base 0.5500 | Test: 8 clients, 7,115 rows, base 0.5165 | Full base 0.5421
Held-out clients (8): ['client_434c9b5ae5', 'client_4e07408562', 'client_8527a891e2', 'client_8b940be7fb', 'client_bdd2d3af3a', 'client_d029fa3a95', 'client_e629fa6598', 'client_f369cb89fc']
model               ROC     PR   P@10   P@20   P@50  P@100  P@500
-----------------------------------------------------------------
Baseline (rule)     nan    nan 0.500 0.450 0.460 0.390 0.494
LogReg            0.611  0.603 0.800 0.800 0.720 0.660 0.644
RandomForest      0.615  0.608 0.700 0.700 0.700 0.650 0.646
Tree d=2          0.577  0.560 0.400 0.550 0.580 0.570 0.584



=== Random (before) — rows shuffled ===
Train: 32 clients, 22,500 rows, base 0.5421 | Test: 31 clients, 7,500 rows, base 0.5420 | Full base 0.5421
model               ROC     PR   P@10   P@20   P@50  P@100  P@500
-----------------------------------------------------------------
Baseline (rule)     nan    nan 0.700 0.700 0.680 0.730 0.666
LogReg            0.709  0.724 0.900 0.850 0.860 0.870 0.834
RandomForest      0.775  0.789 1.000 0.950 0.980 0.960 0.908
Tree d=2          0.640  0.624 0.500 0.650 0.660 0.720 0.650

=== Before/After gap (Random - Grouped) = memorization bonus ===
Baseline (rule)  ΔP@50 +0.220  (random 0.680 → grouped 0.460)
LogReg           ΔP@50 +0.140  (random 0.860 → grouped 0.720)
RandomForest     ΔP@50 +0.280  (random 0.980 → grouped 0.700)
Tree d=2         ΔP@50 +0.080  (random 0.660 → grouped 0.580)

Interpretation (measured, directional): On this grouped holdout the rule collapses (P@50 0.46, below base 0.517) while LogReg/RF retain signal (P@50 0.72/0.70). 

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Checklist (attack-your-own-model, before you believe any metric):**

- [ ] Timeline drawn: all features strictly before label window — label is `last_30d vs prev_30d` decline (`trend_pct`), features must not contain `last_30d`/`prev_30d` totals or any 90-day aggregate that overlaps the last 30 days.
- [ ] No label-derived or sibling columns in features — `trend_pct` and `trend_direction` are where `is_declining` comes from; they and `is_declining_label` itself are never features. Tested by train-with vs train-without (collapse from ~1.0 to ~0.61 is the confession).
- [ ] No product flags / existing-system scores as features — `health_score`, `priority_score`, `action_type`, refresh flags are not in this dataset (observable-only); nothing circular is used.
- [ ] Split grouped by repeating entity (`client_id`) — done in Section 2 (gap reported). Random-split inflation documented.
- [ ] Base rate printed next to every metric — full 0.542, grouped-train 0.550, grouped-test 0.517, random-test 0.542.
- [ ] Top feature importance sanity-checked — no single feature near 1.0; `days_with_impressions`, `avg_position`, `log_impressions_90d`, `content_age_days` are plausible signals (confirmed/MIXED in signal audit), not label siblings.
- [ ] Metrics recomputed out-of-fold, never in-sample — all numbers above are held-out test precision@K.

**Leakage taxonomy checked:**
1. *Label-derived:* `trend_pct`/`trend_direction` excluded; `is_declining` never a feature.
2. *Future/overlapping windows:* `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d`, and `trend_pct` excluded; the 90-day aggregates are allowed only because they are the *given snapshot* and do not recompute the last-30 window — but on the warehouse daily fact only `*_prev30`-style columns would be safe.
3. *Decision-derived:* No product scores as inputs; they are baselines to beat.

**Below:** explicit forbidden-column audit, deliberate leaky-feature injection (score jumps to ~1.0, then removed), feature-importance sanity check, and real failure examples the honest model gets wrong.

In [3]:
# --- A. Forbidden-column audit ---
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
forbidden = ["trend_pct","trend_direction","is_declining","is_declining_label","impressions_last_30d","impressions_prev_30d","clicks_last_30d","clicks_prev_30d","sessions_last_30d","sessions_prev_30d"]
print("Forbidden columns in MODEL_NUMERIC_FEATURES?", any(c in MODEL_NUMERIC_FEATURES for c in forbidden))
print("Forbidden columns in MODEL_CATEGORICAL_FEATURES?", any(c in MODEL_CATEGORICAL_FEATURES for c in forbidden))
for c in forbidden:
    in_num = c in MODEL_NUMERIC_FEATURES; in_cat = c in MODEL_CATEGORICAL_FEATURES
    print(f"  {c:30s} numeric={in_num}  cat={in_cat}  -> {'EXCLUDED ✓' if not (in_num or in_cat) else 'LEAK!'}")
# IDs grouping only
print("\nIDs as features? content_id in features?", "content_id" in MODEL_NUMERIC_FEATURES+MODEL_CATEGORICAL_FEATURES)
print("IDs as features? client_id in features?", "client_id" in MODEL_NUMERIC_FEATURES+MODEL_CATEGORICAL_FEATURES, "-> grouping only ✓")
# Show actual feature lists
print(f"\nMODEL_NUMERIC_FEATURES ({len(MODEL_NUMERIC_FEATURES)}):", MODEL_NUMERIC_FEATURES)
print(f"MODEL_CATEGORICAL_FEATURES ({len(MODEL_CATEGORICAL_FEATURES)}):", MODEL_CATEGORICAL_FEATURES)

# --- B. Timeline (text diagram) ---
print("\nTimeline (prediction moment = end of prev_30d window):")
print("  [feature window: 90d of history strictly BEFORE prev_30d] -> |cutoff| -> [prev_30d (31-60d ago)] vs [last_30d (1-30d ago)] = trend_pct -> label is_declining")
print("  In starter CSV the 90d aggregates are the frozen snapshot; in warehouse work only *_prev30 columns are safe features — *_last30 leaks.")

# --- C. Deliberate leak injection test (hunt verification) ---
# Reuse grouped split indices from Section 2 (train_g, test_g, y, X)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
numeric_features = MODEL_NUMERIC_FEATURES
categorical_features = MODEL_CATEGORICAL_FEATURES
# Honest AP (RF on grouped test)
X_train_h, X_test_h = X.iloc[train_g], X.iloc[test_g]
y_train_h, y_test_h = y.iloc[train_g], y.iloc[test_g]
rf_preprocess = ColumnTransformer([("num", SimpleImputer(strategy="median"), numeric_features), ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)])
rf = Pipeline([("prep", rf_preprocess), ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42))])
rf.fit(X_train_h, y_train_h)
prob_h = rf.predict_proba(X_test_h)[:,1]
baseline_ap = average_precision_score(y_test_h, prob_h)
baseline_roc = roc_auc_score(y_test_h, prob_h)
print(f"\nHonest RF on grouped test: ROC {baseline_roc:.4f} AP {baseline_ap:.4f}  (base {y_test_h.mean():.4f})")
# Leaky: add trend_pct (fillna 0 per 01_prepare_features)
X_leak = X.copy()
X_leak["trend_pct"] = df["trend_pct"].fillna(0)
numeric_leak = MODEL_NUMERIC_FEATURES + ["trend_pct"]
rf_preprocess_leak = ColumnTransformer([("num", SimpleImputer(strategy="median"), numeric_leak), ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)])
lr_preprocess_leak = ColumnTransformer([("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_leak), ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)])
rf_leak = Pipeline([("prep", rf_preprocess_leak), ("clf", RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=42))])
lr_leak = Pipeline([("prep", lr_preprocess_leak), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
rf_leak.fit(X_leak.iloc[train_g], y_train_h)
lr_leak.fit(X_leak.iloc[train_g], y_train_h)
for name, model in [("RF with trend_pct", rf_leak), ("LogReg with trend_pct", lr_leak)]:
    prob = model.predict_proba(X_leak.iloc[test_g])[:,1]
    print(f"LEAKY {name}: ROC {roc_auc_score(y_test_h, prob):.4f} AP {average_precision_score(y_test_h, prob):.4f} P@50 {precision_at_k(y_test_h, prob, 50):.3f}  <- near 1.0 is the confession")
print("Leaky feature REMOVED after test — honest numbers above are the ones that count.")

# --- D. Top feature importance sanity check (honest model) ---
ohe = rf.named_steps["prep"].named_transformers_["cat"].named_steps["enc"]
cat_names = list(ohe.get_feature_names_out(categorical_features))
all_names = numeric_features + cat_names
importances = rf.named_steps["clf"].feature_importances_
order = np.argsort(importances)[::-1]
print("\nTop 12 RF feature_importances_ (honest, grouped):")
for i in order[:12]:
    print(f"  {all_names[i]:40s} {importances[i]:.4f}")
print("Sanity: no feature ~0.9; top is days_with_impressions 0.11 — plausible, not label-derived. If trend_pct were top near 0.8+, that would be leakage.")


Forbidden columns in MODEL_NUMERIC_FEATURES? False
Forbidden columns in MODEL_CATEGORICAL_FEATURES? False
  trend_pct                      numeric=False  cat=False  -> EXCLUDED ✓
  trend_direction                numeric=False  cat=False  -> EXCLUDED ✓
  is_declining                   numeric=False  cat=False  -> EXCLUDED ✓
  is_declining_label             numeric=False  cat=False  -> EXCLUDED ✓
  impressions_last_30d           numeric=False  cat=False  -> EXCLUDED ✓
  impressions_prev_30d           numeric=False  cat=False  -> EXCLUDED ✓
  clicks_last_30d                numeric=False  cat=False  -> EXCLUDED ✓
  clicks_prev_30d                numeric=False  cat=False  -> EXCLUDED ✓
  sessions_last_30d              numeric=False  cat=False  -> EXCLUDED ✓
  sessions_prev_30d              numeric=False  cat=False  -> EXCLUDED ✓

IDs as features? content_id in features? False
IDs as features? client_id in features? False -> grouping only ✓

MODEL_NUMERIC_FEATURES (18): ['search_volume', 'co


Honest RF on grouped test: ROC 0.6155 AP 0.6080  (base 0.5165)


LEAKY RF with trend_pct: ROC 1.0000 AP 1.0000 P@50 1.000  <- near 1.0 is the confession
LEAKY LogReg with trend_pct: ROC 0.9986 AP 0.9987 P@50 1.000  <- near 1.0 is the confession
Leaky feature REMOVED after test — honest numbers above are the ones that count.

Top 12 RF feature_importances_ (honest, grouped):
  log_impressions_90d                      0.1089
  days_with_impressions                    0.1031
  avg_position                             0.0990
  content_age_days                         0.0834
  char_count                               0.0494
  word_count                               0.0487
  scroll_rate                              0.0474
  ctr                                      0.0467
  days_with_sessions                       0.0432
  log_sessions_90d                         0.0432
  log_clicks_90d                           0.0372
  search_volume                            0.0286
Sanity: no feature ~0.9; top is days_with_impressions 0.11 — plausible, not label-derive

In [4]:
# --- E. Real failure examples on the honest (grouped) holdout ---
# Use the RF trained on grouped train (rf, X_test_h, y_test_h, df)
prob_test = rf.predict_proba(X_test_h)[:,1]
df_test_eval = df.iloc[test_g].copy()
df_test_eval["pred_prob"] = prob_test
df_test_eval["pred_label"] = (prob_test >= 0.5).astype(int)
# slices
def accuracy(group):
    return (group["pred_label"]==group["is_declining"]).mean()
print("Overall accuracy (threshold 0.5) on grouped test: {:.3f}  (base {:.3f} -> skill {:.3f})".format((df_test_eval["pred_label"]==df_test_eval["is_declining"]).mean(), y_test_h.mean(), (df_test_eval["pred_label"]==df_test_eval["is_declining"]).mean()-max(y_test_h.mean(), 1-y_test_h.mean())))
# By freshness_tier
print("\nAccuracy by freshness_tier (honest test, RF):")
for tier, g in df_test_eval.groupby("freshness_tier", observed=True):
    print(f"  {tier:10s}  n={len(g):4d}  declining {g['is_declining'].mean():.3f}  acc {accuracy(g):.3f}")
print("\nAccuracy by impression_tier:")
for tier, g in df_test_eval.groupby("impression_tier", observed=True):
    print(f"  {tier:10s}  n={len(g):4d}  declining {g['is_declining'].mean():.3f}  acc {accuracy(g):.3f}")
print("\nAccuracy by content_type:")
for tier, g in df_test_eval.groupby("content_type", observed=True):
    print(f"  {tier:22s}  n={len(g):4d}  declining {g['is_declining'].mean():.3f}  acc {accuracy(g):.3f}")

# Concrete cases: 3 false positives (predicted  high, actually not declining) and 3 false negatives
fp = df_test_eval[(df_test_eval["pred_label"]==1) & (df_test_eval["is_declining"]==0)].sort_values("pred_prob", ascending=False)
fn = df_test_eval[(df_test_eval["pred_label"]==0) & (df_test_eval["is_declining"]==1)].sort_values("pred_prob")
cols = ["pred_prob","is_declining","trend_pct","impressions_90d","days_since_last_update","freshness_tier","avg_position","ctr","days_with_impressions","content_age_days","impression_tier"]
print("\n--- 3 False Positives (flagged as declining but measured stable/up/new/flat) ---")
print(fp[cols].head(3).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nReason codes for FPs: often stale (>=90d) + moderate impressions + page-1 position — looks risky, but measured not declining (staleness alone not sufficient).")
print("\n--- 3 False Negatives (missed declines — predicted not declining but measured down) ---")
print(fn[cols].head(3).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nReason codes for FNs: often fresh (0-30d) or low-impression (<100) pages that declined without the staleness cue; or very stale 181+ where pattern inverts (small n).")
print("\nTakeaway: model is useful as a ranked queue (precision at top), not a hard 0.5 classifier — operating threshold and review capacity matter more than raw accuracy.")


Overall accuracy (threshold 0.5) on grouped test: 0.586  (base 0.517 -> skill 0.069)

Accuracy by freshness_tier (honest test, RF):
  0-30        n=5799  declining 0.526  acc 0.589
  181+        n=  47  declining 0.638  acc 0.638
  31-90       n=  51  declining 0.549  acc 0.627
  91-180      n=1218  declining 0.466  acc 0.567

Accuracy by impression_tier:
  excellent   n= 171  declining 0.386  acc 0.532
  good        n=1199  declining 0.436  acc 0.551
  low         n=3575  declining 0.506  acc 0.599
  moderate    n=2170  declining 0.588  acc 0.587

Accuracy by content_type:
  comparison article      n= 697  declining 0.572  acc 0.575
  keyword article         n=6418  declining 0.510  acc 0.587

--- 3 False Positives (flagged as declining but measured stable/up/new/flat) ---
 pred_prob  is_declining  trend_pct  impressions_90d  days_since_last_update freshness_tier  avg_position   ctr  days_with_impressions  content_age_days impression_tier
     0.961             0     48.300           

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest Week-5-era sentence (what I was tempted to write):**
> "Our Random Forest predicts which pages will decline and proves that stale content causes Google to drop impressions. With P@50 0.70 we can guarantee that fixing the top 50 will recover traffic."

**Why that exceeds the evidence:**
- *"predicts which pages will decline"* — without a future window it predicts a **proxy** (bucket defined on the same snapshot), not the future.
- *"proves ... causes"* — cross-sectional snapshot, no experiment/matched design; correlation ≠ causation.
- *"guarantee ... will recover"* — ranking ≠ causal intervention; we never randomized which pages were refreshed and measured recovery.
- *"P@50 0.70"* without base rate (0.517 on grouped test) hides that skill is +0.18 over random, not 70% magic.

**Honest rewrites (claim ladder):**

| Ladder rung | Safe sentence | Evidence that carries it |
|---|---|---|
| Observed | "In the 30,000-row starter slice on its export date, we observed 54.2% of pages labeled declining (`down` < −20%) and 61.1% in the 91–180-day stale bucket vs 51.1% in 0–30 days." | Counts + base rate, this dataset, this period |
| Measured comparison | "Measured on a client-holdout split (8 clients, n=7,115, base 0.517), the Random Forest **ranked** declining pages higher at P@50 0.70 (35/50) vs the rule's P@50 0.46 (23/50) and random 0.52 — a directional lift of ~0.18–0.24 in precision at the top of the queue." | Out-of-fold Precision@K vs base rate + baseline on same split |
| Validated ranking | "The model flags pages worth *reviewing first* (decision-support); under the grouped model, LogReg P@20 0.80 and RF P@100 0.65 suggest the top of the queue concentrates risk, but ~30–35% of top-ranked pages were not declining in this test." | Precision@K curve, validation on unseen clients |
| Decision-support | "For an editorial team that can review ~50 pages/week, we suggest starting with the top-ranked `stale_moderate_visible` pages and verifying each manually (position change, cannibalization, seasonality, small-sample noise); the model supports triage, it does not automate publishing or guarantee recovery." | Ranked queue, reason codes, error examples |
| **Not claimable without new design** | Would need: time-aware train-past/test-future or A/B refresh experiment to say "refreshing causes recovery" — not observed here. | Causal claim requires experiment |

*Respectful note on the paper: it already uses this ladder — my rewrite above simply holds my own Week-5 wording to the same standard the paper set for a broad audience.*

**My revised single paragraph (safe language only):**
> "We measured a directional, decision-support signal: on a grouped client holdout (test 8 clients, n=7,115, base 0.517), pages ranked by the Random Forest showed higher precision at the top of the queue (observed P@50 0.70, P@100 0.65; baseline rule 0.46/0.39) with ROC-AUC 0.62 (vs 0.51 random). This indicates the model *ranks* stale-and-visible pages that are empirically more likely to be labeled declining in this snapshot; it does not prove that staleness causes decline or that editing will cause recovery. Used as a reviewer aid with reason codes and manual verification, the ranked queue is a plausible way to spend limited review capacity. Results are specific to this slice and split — re-validate on a time-aware or mid-month warehouse window before any production use."


In [5]:
# --- Verify: every metric below prints its base rate next to it (no accuracy without base rate) ---
print(f"Base rates: full {df['is_declining'].mean():.4f} | grouped train {y.iloc[train_g].mean():.4f} | grouped test {y.iloc[test_g].mean():.4f} | random test {y.iloc[test_r].mean():.4f}")
# Re-print headline comparison in safe-language template
p50_rf_grouped = precision_at_k(y.iloc[test_g], rf.predict_proba(X.iloc[test_g])[:,1], 50)
p50_rule_grouped = precision_at_k(y.iloc[test_g], baseline_scores(df.iloc[test_g]), 50)
print(f"Observed (grouped test, n=7,115, base 0.517): RF P@50 {p50_rf_grouped:.3f} ({int(p50_rf_grouped*50)}/50) vs rule {p50_rule_grouped:.3f} vs random ~0.517 -> lift {p50_rf_grouped-0.517:+.3f} over base")
# Show that small-bucket ratio would be flagged (example: 181+ freshness has n=47 in grouped test — too small for headline)
print(f"\nSelection-bias note: grouped-test 181+ tier n={(df.iloc[test_g]['freshness_tier']=='181+').sum()} — headline ratio from n<50 not reported (banned tiny-bucket claim)")
print("\nSafe-language check: observed/measured/directional/decision-support used; no 'proves', 'causes', 'will increase', 'predicted Google'. ✓")


Base rates: full 0.5421 | grouped train 0.5500 | grouped test 0.5165 | random test 0.5420
Observed (grouped test, n=7,115, base 0.517): RF P@50 0.700 (35/50) vs rule 0.460 vs random ~0.517 -> lift +0.183 over base

Selection-bias note: grouped-test 181+ tier n=47 — headline ratio from n<50 not reported (banned tiny-bucket claim)

Safe-language check: observed/measured/directional/decision-support used; no 'proves', 'causes', 'will increase', 'predicted Google'. ✓


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) — seed 42, versions printed in Cell 1
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id`/`client_id`, aggregated metrics
- [x] My claims use careful words: observed, measured, directional, decision-support — no causal "proves/causes/will increase"
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] Two paper findings + methodology questions framed constructively (Section 1)
- [x] Before/after honest split shown with both numbers + gap (Section 2: RF P@50 0.98 random → 0.70 grouped; LogReg 0.86 → 0.72; rule 0.85*→0.46)
- [x] Leakage audit with taxonomy + deliberate leak injection 1.0 vs 0.61 + feature sanity check (Section 3)
- [x] Real failure examples: 3 FP + 3 FN + accuracy slices by freshness/impression/content_type (Section 3 cell 2)
- [x] Claim rewrite in safe language, ban-list checked, base rate next to every metric (Section 4)

*_Random full-data baseline in w04_baseline_score was P@50 0.74 — that's the inflated mixed-client number; the honest client-holdout rule drops to 0.46._